# Part 1: One qubit

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 1.1 Bits and qubits

### Measuring a qubit, shot by shot

A random-number generator that uses the Born-rule probabilities gives outcomes with exactly the same statistics as measuring the qubit. `rng.choice([0, 1], size=N, p=probs)` returns N outcomes, one per shot.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=1)
alpha, beta = np.sqrt(3) / 2, 1 / 2          # the state (√3/2)|0> + (1/2)|1>
probs = [abs(alpha) ** 2, abs(beta) ** 2]    # Born rule
print("P(0), P(1) =", np.round(probs, 4))

for N in [100, 10_000]:
    outcomes = rng.choice([0, 1], size=N, p=probs)   # N shots
    p1 = outcomes.mean()                              # fraction of 1s
    error = np.sqrt(p1 * (1 - p1) / N)                # standard error
    print(f"N = {N:6d}:  P(1) is about {p1:.3f} ± {error:.4f}")

The estimates scatter around the true value P(1) = 0.25, and the error bar shrinks about tenfold when the number of shots grows a hundredfold. Change `seed` to get a different run.

### Exercise 1.1.1: Watch the error shrink like 1/√N

For N = 100, 400 and 1,600 shots, repeat the whole experiment 2,000 times and measure how much the estimates of P(1) spread out, as their standard deviation. Compare with √(p(1 − p)/N) for p = 0.25.

<details><summary>Hint</summary>

`rng.binomial(N, p, size=2000)` counts the 1s in 2,000 experiments at once. Divide by N to get the estimates, and use `.std()` for the spread.

</details>

In [ ]:
# Your code here

### Exercise 1.1.2: A superposition is not a hidden coin

Simulate 1,000 runs of "apply H, then measure" for two preparations: the state |+⟩, and a hidden coin that is |0⟩ or |1⟩ with probability 1/2 each. Count how often each one gives 0.

<details><summary>Hint</summary>

For the hidden coin, choose |0⟩ or |1⟩ at random in every run, apply H, then measure with the Born rule.

</details>

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=3)
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
ket0, ket1 = np.array([1, 0]), np.array([0, 1])

def measure(psi):
    p = np.abs(psi) ** 2
    return rng.choice([0, 1], p=p / p.sum())

# your code: 1,000 runs for |+>, then 1,000 runs for the hidden coin

## 1.2 Phase

### Global phase changes nothing, relative phase does

A measurement in the X basis is H followed by an ordinary measurement, so its probabilities are `abs(H @ psi) ** 2`. The four states below have identical 0/1 probabilities.

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
s = 1 / np.sqrt(2)
states = {
    "|+>":          np.array([s, s]),
    "e^(0.7i)|+>":  np.exp(0.7j) * np.array([s, s]),
    "|+i>":         np.array([s, 1j * s]),
    "|->":          np.array([s, -s]),
}
for name, psi in states.items():
    pz = np.abs(psi) ** 2          # Z basis: P(0), P(1)
    px = np.abs(H @ psi) ** 2      # X basis: P(+), P(-)
    print(f"{name:12s}  Z basis {pz.round(3)}   X basis {px.round(3) + 0}")

Multiplying |+⟩ by e<sup>0.7i</sup> changes no probability in either basis. Changing the relative phase moves the X-basis odds from a certain + (|+⟩), through 50/50 (|+i⟩), to a certain − (|−⟩).

### Exercise 1.2.1: The relative phase sets P(+)

For the state (|0⟩ + e<sup>iφ</sup>|1⟩)/√2, compute P(+) for φ = 0, π/4, π/2, 3π/4 and π, and compare with the formula (1 + cos φ)/2.

<details><summary>Hint</summary>

P(+) is the first entry of `abs(H @ psi) ** 2`. `np.linspace(0, np.pi, 5)` gives the five angles.

</details>

In [ ]:
# Your code here

### Exercise 1.2.2: Remove the global phase

Write `remove_global_phase(psi)`, which multiplies a state by a phase factor so that its first amplitude becomes real and non-negative. Use it to show that (i|0⟩ + |1⟩)/√2 is the state |−i⟩, and print the relative phase.

<details><summary>Hint</summary>

The angle of α is `np.angle(psi[0])`, and multiplying by `np.exp(-1j * angle)` turns it back to zero. The relative phase is `np.angle(beta) - np.angle(alpha)`.

</details>

In [ ]:
# Your code here

## 1.3 The Bloch sphere

### From amplitudes to a point on the sphere

The Bloch vector is (⟨X⟩, ⟨Y⟩, ⟨Z⟩), so one short function finds it for any state. Here it places the six landmark states.

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def bloch(psi):
    """The Bloch vector (<X>, <Y>, <Z>) of a one-qubit state."""
    return np.array([np.vdot(psi, P @ psi).real for P in (X, Y, Z)])


s = 1 / np.sqrt(2)
landmarks = {"|0>": [1, 0], "|1>": [0, 1], "|+>": [s, s], "|->": [s, -s], "|+i>": [s, 1j * s], "|-i>": [s, -1j * s]}
for name, amps in landmarks.items():
    print(f"{name:5s} ->", bloch(np.array(amps)).round(3) + 0)

Each state lands on the axis given in the chapter's table: |0⟩ and |1⟩ at the poles, |±⟩ on the x axis and |±i⟩ on the y axis.

### Exercise 1.3.1: Angles in, state out

Write `state(theta, phi)`, which returns cos(θ/2)|0⟩ + e<sup>iφ</sup> sin(θ/2)|1⟩. For three random pairs of angles, check that the Bloch vector of `state(theta, phi)` is (sin θ cos φ, sin θ sin φ, cos θ).

<details><summary>Hint</summary>

Reuse the `bloch` function from the example above, and compare with `np.allclose`.

</details>

In [ ]:
# Your code here

### Exercise 1.3.2: The overlap rule

Draw two random states, compute |⟨ψ|χ⟩|² directly and from their Bloch vectors r and s with the rule (1 + r·s)/2. Then try |0⟩ and |1⟩, which sit at opposite poles.

<details><summary>Hint</summary>

A random state is any random complex column divided by its length. `r @ s` is the dot product of two Bloch vectors.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 1.1.1: Watch the error shrink like 1/√N

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=2)
p = 0.25
for N in [100, 400, 1600]:
    estimates = rng.binomial(N, p, size=2000) / N
    formula = np.sqrt(p * (1 - p) / N)
    print(f"N = {N:5d}:  spread {estimates.std():.4f}   formula {formula:.4f}")

Expected output:

```
N =   100:  spread 0.0437   formula 0.0433
N =   400:  spread 0.0223   formula 0.0217
N =  1600:  spread 0.0109   formula 0.0108
```

Each fourfold increase in N halves the formula's value, and the measured spread follows it closely, roughly halving too.

### Solution 1.1.2: A superposition is not a hidden coin

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=3)
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
ket0, ket1 = np.array([1, 0]), np.array([0, 1])
plus = H @ ket0


def measure(psi):
    p = np.abs(psi) ** 2
    return rng.choice([0, 1], p=p / p.sum())


zeros_plus = sum(measure(H @ plus) == 0 for _ in range(1000))
zeros_coin = 0
for _ in range(1000):
    hidden = ket0 if rng.random() < 0.5 else ket1
    zeros_coin += measure(H @ hidden) == 0

print("|+>, then H:         ", zeros_plus, "zeros out of 1000")
print("hidden coin, then H: ", zeros_coin, "zeros out of 1000")

Expected output:

```
|+>, then H:          1000 zeros out of 1000
hidden coin, then H:  508 zeros out of 1000
```

### Solution 1.2.1: The relative phase sets P(+)

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
for phi in np.linspace(0, np.pi, 5):
    psi = np.array([1, np.exp(1j * phi)]) / np.sqrt(2)
    p_plus = abs((H @ psi)[0]) ** 2
    formula = (1 + np.cos(phi)) / 2
    print(f"phi = {phi:.4f}   P(+) = {p_plus:.4f}   (1 + cos phi)/2 = {formula + 0:.4f}")

Expected output:

```
phi = 0.0000   P(+) = 1.0000   (1 + cos phi)/2 = 1.0000
phi = 0.7854   P(+) = 0.8536   (1 + cos phi)/2 = 0.8536
phi = 1.5708   P(+) = 0.5000   (1 + cos phi)/2 = 0.5000
phi = 2.3562   P(+) = 0.1464   (1 + cos phi)/2 = 0.1464
phi = 3.1416   P(+) = 0.0000   (1 + cos phi)/2 = 0.0000
```

### Solution 1.2.2: Remove the global phase

In [ ]:
import numpy as np


def remove_global_phase(psi):
    return psi * np.exp(-1j * np.angle(psi[0]))


psi = np.array([1j, 1]) / np.sqrt(2)
clean = remove_global_phase(psi)
minus_i = np.array([1, -1j]) / np.sqrt(2)
print("without the global phase:", clean.round(4))
print("relative phase:", np.angle(clean[1]) / np.pi, "x pi")
print("equal to |-i>:", np.allclose(clean, minus_i))

Expected output:

```
without the global phase: [0.7071+0.j     0.    -0.7071j]
relative phase: -0.5 x pi
equal to |-i>: True
```

### Solution 1.3.1: Angles in, state out

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def bloch(psi):
    return np.array([np.vdot(psi, P @ psi).real for P in (X, Y, Z)])


def state(theta, phi):
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])


rng = np.random.default_rng(seed=4)
for _ in range(3):
    theta, phi = rng.uniform(0, np.pi), rng.uniform(0, 2 * np.pi)
    r = bloch(state(theta, phi))
    expected = [np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)]
    print(f"theta = {theta:.2f}, phi = {phi:.2f}:  r = {r.round(3)}  matches: {np.allclose(r, expected)}")

Expected output:

```
theta = 2.96, phi = 3.21:  r = [-0.177 -0.013 -0.984]  matches: True
theta = 3.07, phi = 0.51:  r = [ 0.065  0.036 -0.997]  matches: True
theta = 1.91, phi = 2.37:  r = [-0.673  0.661 -0.331]  matches: True
```

### Solution 1.3.2: The overlap rule

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def bloch(psi):
    return np.array([np.vdot(psi, P @ psi).real for P in (X, Y, Z)])


def random_state(rng):
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    return v / np.linalg.norm(v)


rng = np.random.default_rng(seed=5)
for psi, chi in [(random_state(rng), random_state(rng)), (np.array([1, 0]), np.array([0, 1]))]:
    direct = abs(np.vdot(psi, chi)) ** 2
    rule = (1 + bloch(psi) @ bloch(chi)) / 2
    print(f"|<psi|chi>|^2 = {direct:.4f}    (1 + r.s)/2 = {rule:.4f}")

Expected output:

```
|<psi|chi>|^2 = 0.7701    (1 + r.s)/2 = 0.7701
|<psi|chi>|^2 = 0.0000    (1 + r.s)/2 = 0.0000
```

Opposite points on the sphere have r·s = −1, so their overlap is 0: they are orthogonal states.